# Pre-trip ETA baseline

Train a zone-level duration model using only pickup zone, destination zone, and local departure time. Compare it with a mean predictor and the saved retrospective model, re-evaluated on the current validation/test rows. Historical scores are also shown separately. The old model uses actual distance and final rate code; its score is a reference, not an attainable pre-trip accuracy promise.

Hypothesis: zone and time features beat a constant predictor on a chronological validation split. Expect higher error than the retrospective model. Use the current processed data and original split rule; historical training-row provenance is unavailable, and this does not test broader trip coverage.

In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/config.yaml").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook from within the project.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.pretrip import train, predict


## Train reproducibly

Settings come from `configs/config.yaml`, including seed 42 and the original XGBoost baseline hyperparameters. Validation controls early stopping; the final seven days remain test data. Both training and inference call the same feature builder. Save a separate model bundle containing the fitted estimator and geographic mapping.

This cell trains on the full processed January dataset and overwrites only the pre-trip experiment artifacts.

In [2]:
report = train()

Split sizes: {'train': 1529266, 'val': 684997, 'test': 678572}
{
  "pretrip_xgboost": {
    "val": {
      "mae": 3.856947949448979,
      "rmse": 5.680893036548132,
      "r2": 0.7300123614499242
    },
    "test": {
      "mae": 3.9594097001262374,
      "rmse": 5.894010814726693,
      "r2": 0.7039564881400284
    }
  },
  "dummy_mean": {
    "val": {
      "mae": 7.745733001171714,
      "rmse": 10.93322833635486,
      "r2": -1.7252318636584718e-05
    },
    "test": {
      "mae": 7.73362427661047,
      "rmse": 10.833320134098912,
      "r2": -0.00013089881529793068
    }
  },
  "retrospective_saved_model": {
    "val": {
      "mae": 3.485176886215411,
      "rmse": 5.139807597889107,
      "r2": 0.7789938393302548
    },
    "test": {
      "mae": 3.59040329024115,
      "rmse": 5.351462872921631,
      "r2": 0.7559500493491128
    }
  }
}


## Compare accuracy in minutes

The historical FLAML model was tuned and uses additional, post-trip inputs, so this comparison is not a controlled feature ablation. The new model must beat the mean predictor on validation; the test set is for final reporting.

In [3]:
records = []
for name, splits in report["metrics"].items():
    for split, scores in splits.items():
        records.append({"model": name, "split": split, **scores})
reference = report["historical_reference"]
if reference:
    for split in ("val", "test"):
        records.append({"model": reference["model_name"] + " (historical, post-trip inputs)", "split": split, **reference[split]})
comparison = pd.DataFrame(records)
print(comparison.round(4).to_string(index=False))
print("Beats mean baseline on validation:", report["beats_dummy_on_validation"])

                                       model split    mae    rmse      r2
                             pretrip_xgboost   val 3.8569  5.6809  0.7300
                             pretrip_xgboost  test 3.9594  5.8940  0.7040
                                  dummy_mean   val 7.7457 10.9332 -0.0000
                                  dummy_mean  test 7.7336 10.8333 -0.0001
                   retrospective_saved_model   val 3.4852  5.1398  0.7790
                   retrospective_saved_model  test 3.5904  5.3515  0.7560
flaml_xgboost (historical, post-trip inputs)   val 2.4651  3.9683  0.8661
flaml_xgboost (historical, post-trip inputs)  test 2.5740  4.2092  0.8467
Beats mean baseline on validation: True


## Predict before departure

Supply known TLC zone IDs and a timezone-naive NYC local departure time. No actual distance, rate code, passenger count, or manually engineered columns are needed. This is an illustrative January request, not a live-traffic forecast.

In [4]:
request = pd.DataFrame([{
    "PULocationID": 161,
    "DOLocationID": 141,
    "pickup_datetime": "2023-01-25 08:00:00",
}])
print(f"Estimated trip duration: {predict(request)[0]:.2f} minutes")

Estimated trip duration: 11.62 minutes


## Limitations and next steps

- This experiment keeps the existing filtered cohort for comparison; it does not establish performance on excluded trips.
- The saved bundle persists the zone map to prevent encoding drift between training and prediction.
- The airport flag retains the earlier lookup rule, which excludes the EWR service-zone label.
- Validate on a later month and inspect errors by origin/destination and departure time.
- If useful, evaluate estimated routing distances with the same method in training and inference.
- The existing SHAP results describe the old model; explainability needs to be repeated for this model.
- FastAPI, monitoring, and automated retraining remain future work.